# Phase 2: novelty and drift

Release `data/results/2026-09-23/`, earnings calls only; the JPMorgan event call is neither a target call nor part of any reference set. Plan: `plans/phase_2.md`. Computation: `analysis/phase2_novelty.py`, `analysis/phase2_inference.py`, `analysis/phase2_labelling.py`; outputs in `data/phase2/`.

Sentence novelty is 1 minus the maximum cosine similarity (masked text, `all-mpnet-base-v2`) to a random sample of 415 sentences from the same bank's earlier calls, averaged over 20 draws. **Results are provisional** until the novelty labelling sample is returned.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase2"
COLORS = {"UBS": "#2a78d6", "JPM": "#eb6834"}
MARKERS = {"UBS": "o", "JPM": "s"}
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.precision", 4)

In [ ]:
meta = json.loads((OUT / "models.json").read_text())
short = json.loads((OUT / "phase2_meta.json").read_text())["short_share"]
pd.DataFrame(short).rename_axis("bank").round(3).assign(reference_size_R=meta["reference_sizes"]["mpnet"])

Sentences of three words or fewer are excluded: 4 % of UBS and 12 % of JPMorgan management sentences, and 29 % and 35 % of analyst sentences, mostly greetings and thanks. The reference sample has 415 sentences, the size of the smaller 2023-Q1 call.

## P2-2: decay of management novelty (primary)

In [ ]:
calls = pd.read_csv(OUT / "call_novelty.csv")
prim = calls[calls.variant == "primary"]
fig, ax = plt.subplots(figsize=(8, 4))
for bank, g in prim.groupby("bank"):
    g = g.sort_values("t")
    ax.plot(g.quarter, g.value, marker=MARKERS[bank], color=COLORS[bank], lw=2, ms=8, label=bank)
    b, a = np.polyfit(g.t, g.value, 1)
    ax.plot(g.quarter, a + b * g.t, color=COLORS[bank], lw=1, ls="--")
    ax.annotate(f"{bank}: slope {b:+.4f} per quarter", (g.quarter.iloc[-1], a + b * 6), xytext=(6, 0),
                textcoords="offset points", va="center", color="0.2", fontsize=9)
ax.set_ylabel("Mean management sentence novelty\n(1 - max cosine similarity)")
ax.set_xlabel("Reporting quarter of the earnings call")
ax.grid(axis="y", color="0.9"); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Management novelty per call against earlier calls, with OLS trend (dashed)", loc="left", fontsize=10)
ax.set_xlim(-0.3, 8.2)
plt.tight_layout()

In [ ]:
res = pd.read_csv(OUT / "p2_decay_and_level.csv")
res["role"] = ["primary", "robustness (paired test)"] + ["robustness"] * 7
res[["variant", "role", "slope_UBS", "slope_JPM", "T_decay", "p_decay", "n_splits"]]

In [ ]:
reg = pd.read_csv(ROOT / "notebooks" / "roman" / "plans" / "test_register.csv")
reg

UBS management novelty falls by 0.0046 per quarter while JPMorgan's rises by 0.0054, so T = −0.0100 (UBS minus JPM) with exact p = 0.013 and a Holm-adjusted p of 0.040 across the three primary tests. The direction is the opposite of the hypothesis: by this measure the merger does not echo longer in UBS communication than the First Republic deal does at JPMorgan.

In [ ]:
same_scale = res[~res.variant.isin(["tfidf", "share_below_0.5"])]
pd.DataFrame({"T_decay primary": [res.T_decay[0]],
              "min over embedding variants": [same_scale.T_decay.min()],
              "max over embedding variants": [same_scale.T_decay.max()],
              "variants with T < 0": [f"{(res.T_decay < 0).sum()} of {len(res)}"],
              "variants with p < 0.05": [f"{(res.p_decay < 0.05).sum()} of {len(res)}"]})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
r = res.iloc[::-1]
ax.axvline(0, color="0.6", lw=1)
ax.scatter(r.T_decay, r.variant, color=np.where(r.p_decay < 0.05, "#2a78d6", "white"),
           edgecolor="#2a78d6", s=60, zorder=3)
ax.set_xlabel("T = slope UBS minus slope JPM (novelty per quarter)")
ax.set_title("Effect size across robustness variants (filled: raw p < 0.05)", loc="left", fontsize=10)
ax.grid(axis="x", color="0.9"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

All nine variants give a negative T, ranging from −0.0111 to −0.0054 per quarter on the embedding scale, so the sign is stable. Significance is not: the constant-speaker variant (p = 0.32), TF-IDF (0.68), the previous-call reference (0.10), the share of novel sentences (0.67) and the paired test (0.063) do not reach 0.05, and by the plan the result is therefore fragile.

The constant-speaker variant is the control for the UBS CFO change after 2023-Q2 and for Dimon's absences; its T of −0.0054 is half the primary value and not significant. Part of the primary effect may thus come from changes in who speaks rather than from what management says.

## Where the difference comes from (exploratory, E2)

In [ ]:
e2 = pd.read_csv(OUT / "e2_curves.csv")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, grp in zip(axes, ["management prepared", "management qa", "analyst"]):
    for bank, g in e2[e2.group == grp].groupby("bank"):
        ax.plot(g.quarter.str.replace("20", "", n=1), g.novelty, marker=MARKERS[bank], color=COLORS[bank],
                lw=2, ms=6, label=bank)
    ax.set_title(grp, loc="left", fontsize=10); ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", color="0.9"); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Mean sentence novelty")
axes[0].legend(frameon=False)
fig.supxlabel("Reporting quarter", fontsize=10)
plt.tight_layout()

In [ ]:
e2.pivot_table(index="quarter", columns=["bank", "group"], values="sentences").fillna(0).astype(int)

*Exploratory.* The UBS decline sits in the prepared remarks, whose novelty falls from 0.41 to about 0.36 as the post-merger slide script settles, while UBS Q&A novelty stays flat. JPMorgan prepared remarks are far less novel (about 0.25) than its Q&A (about 0.48), and its share of Q&A sentences varies strongly between calls, so part of the rising JPMorgan curve is a change in the mix of sections rather than in content.

## E6: prepared remarks and Q&A weighted equally (exploratory, fixed after seeing the results)

In [ ]:
e6 = pd.read_csv(OUT / "e6_call_values.csv")
e6.pivot(index="quarter", columns="bank", values=["prepared", "qa", "value"]).round(3)

In [ ]:
pd.concat([res.loc[[0], ["slope_UBS", "slope_JPM", "T_decay", "p_decay", "level_UBS", "level_JPM", "T_level", "p_level"]]
             .assign(call_value="all management sentences (P2-2)"),
           pd.read_csv(OUT / "e6_sections_equal.csv")[["slope_UBS", "slope_JPM", "T_decay", "p_decay", "level_UBS",
                                                       "level_JPM", "T_level", "p_level"]]
             .assign(call_value="prepared and Q&A weighted equally (E6)")]).set_index("call_value")

*Exploratory, defined after the P2-2 result was known.* With prepared remarks and Q&A weighted equally, the slope difference shrinks from −0.0100 to −0.0039 per quarter and p rises to 0.70. Most of the primary decay difference therefore comes from the changing mix of sections at JPMorgan, not from a change in what is said within a section.

## E1: level of novelty (exploratory)

In [ ]:
res[["variant", "level_UBS", "level_JPM", "T_level", "p_level"]]

*Exploratory.* The primary level difference is −0.009 (p = 0.27), and the sign changes across variants (from −0.016 to +0.021 on the embedding scale, larger with TF-IDF and the share of novel sentences), so no level difference can be claimed. This is consistent with the decision to treat the level as house style.

## E3: what is new (exploratory)

In [ ]:
top = pd.read_csv(OUT / "e3_top_novel.csv")
top[top.quarter.isin(["2023-Q2", "2024-Q3"])].groupby(["bank", "quarter"]).head(5)[
    ["bank", "quarter", "novelty", "plain", "nearest_quarter", "nearest_text"]]

*Exploratory.* The most novel sentences are mostly short conversational remarks (“One and one makes two.”, “It’s sitting in store.”) rather than new disclosures; content such as the non-core run-off profile appears only occasionally. The measure therefore mixes style with substance, which the novelty labelling sample is designed to quantify.

In [ ]:
pd.read_csv(OUT / "e3_capital_share.csv")

*Exploratory.* Among novel UBS management sentences the share mentioning capital-regulation terms doubles from 2.6 % in 2023 to 5.4 % in 2024, while it stays at about 2.5 % at JPMorgan; the counts behind these shares are small (6 and 12 sentences at UBS).

## E4: analyst novelty (exploratory)

In [ ]:
pd.read_csv(OUT / "e4_analysts.csv")

*Exploratory.* Analyst questions show the same pattern in weaker form: UBS slope −0.0015 against +0.0037 at JPMorgan (T = −0.0053, p = 0.45). Analysts at UBS ask less novel questions than at JPMorgan throughout (0.44 against 0.47, p = 0.001).

## E5: consistency with the drift of phase 0 (exploratory)

In [ ]:
drift0 = pd.read_csv(ROOT / "notebooks" / "roman" / "data" / "phase0" / "drift_reference.csv")[["view", "bank", "rho", "p"]]
drift0.merge(pd.read_csv(OUT / "e5_drift_embeddings.csv"), on=["view", "bank"]).rename(
    columns={"rho": "rho_tfidf_phase0", "p": "p_tfidf_phase0"})

*Exploratory.* On sentence embeddings, call similarity still falls with lag in all four views, but much more weakly than with TF-IDF (rho −0.23 to −0.29, none significant). The strong lexical drift of phase 0 is thus largely a change of words rather than of meaning.

## Reading of phase 2

P2-2 is formally significant after Holm (p = 0.040), but its sign is against the hypothesis and by the plan it is fragile: the constant-speaker control and four other variants do not reach 0.05. The decomposition points to format and speaker change rather than content: the UBS decline sits in the settling slide script of the prepared remarks, the JPMorgan rise largely disappears once sections are weighted equally (E6), and the constant-speaker variant halves the effect.

For the supervisory question the result is negative: these data give no evidence that the Credit Suisse merger kept UBS communication novel for longer than the First Republic acquisition did at JPMorgan.

## Validation sample

`data/labelling/novelty.csv` holds 100 management sentences, 12 or 13 per bank and novelty quartile, each with its most similar earlier sentence, blind and in random order; scores and strata are in `novelty_key.csv`. If the AUC between novelty and the label is below 0.7 in either bank, or differs by more than 0.1 between banks, P2-2 and E1 are reported as not interpretable.